In [36]:
library(tidyverse)
library(ggpubr)
library(logging)

source('../scripts/r_funcs.r')
options(readr.show_col_types = FALSE)

theme_set(theme_pubr())
logging::basicConfig()

## TMB boxplot

- TMB: based on T-N paired, fresh, non-LN samples; mutation based on pyclone results (t_alt_count >= 3, t_depth >= 30, autosome mutations)

In [37]:
f_tmb <- '../data/tmb-tn_pair.tsv'
target_size_mb <- 38  # target exome size in megabases

tmb <- read_tsv(f_tmb, show_col_types = F)
loginfo('%g mutations, %g samples', sum(tmb$n_mutation), length(unique(tmb$sample_id)))

2025-12-17 08:51:21.988421 INFO::7994 mutations, 73 samples


In [41]:
# sample filtering
tmb_flt <- tmb %>% 
  filter(
    pair_cat == 'Fresh',
    !grepl('-LN', sample_id),
    sample_type == 'Baseline',
    !is.na(mandard_group)
  ) %>%
  mutate(
    response = factor(case_match(mandard_group, 'good' ~ 'R', 'poor' ~ 'NR'), levels = c('R', 'NR')),
    TMB = n_mutation / (target_size_mb)
  )

loginfo('tn-pair, fresh, non-LN, pre: %g mutations, %g samples', sum(tmb_flt$n_mutation), length(unique(tmb_flt$sample_id)))
tmb_flt %>% 
  select(patient_id, response) %>% 
  unique() %>% 
  count(response)

2025-12-17 08:54:00.793106 INFO::tn-pair, fresh, non-LN, pre: 5099 mutations, 41 samples


response,n
<fct>,<int>
R,27
NR,14


In [43]:
# pre: R vs NR
p <- tmb_flt %>% 
    cell_comp_boxplot(x = 'response', y = 'TMB', xorder = NULL,
                      pt_fill = 'response', fill_order = NULL, pair_by = NULL,
                      facet_by = NULL, ytitle = 'TMB') +
    scale_y_continuous(trans = 'log10') + 
    stat_compare_means(comparisons = list(c('R', 'NR'))) +
    labs(fill = '', x = '', y = 'TMB (Mutations/Mb)', title = 'TMB of Pre-Treatment Samples') 
ggsave(filename = '../box_tmb-pre-R_vs_NR-tn_pair-fresh-t.pdf', plot = p, width = 3.5, height = 5)

Warning message in wilcox.test.default(c(1.18439253577352, 0.754670154534121, -0.048304679574555, :
"cannot compute exact p-value with ties"
